# 00. 记忆分类与工作记忆

Agent 的记忆系统可以按两个维度分类：

| | 短期 | 长期 |
|---|------|------|
| **情节** (发生了什么) | 当前 session 的 tool call trace | 过去任务的成功/失败经验 |
| **语义** (知道什么) | 当前任务的工作状态 | 领域知识、用户偏好 |
| **程序** (怎么做) | — | 可复用的 skill、workflow |

> **检查点**：能举例说明三种记忆类型的区别。

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / 'xmake.lua').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'agents'))

from scripts.nb_helpers import setup, chat, chat_json, chat_with_tools, show_trace
client = setup()


🌐 API 模式 — model=deepseek-v4-flash  base_url=https://api.deepseek.com


In [2]:
# 工作记忆：容量有限的 dict-based store
from dataclasses import dataclass

@dataclass
class WorkingMemory:
    capacity: int = 5
    def __post_init__(self): self.store = {}
    
    def put(self, key: str, value: str):
        if len(self.store) >= self.capacity and key not in self.store:
            oldest = min(self.store.keys(), key=lambda k: self.store[k][1])
            print(f"  Evicted: {oldest}")
            del self.store[oldest]
        self.store[key] = (value, len(self.store))
    
    def get(self, key: str) -> str | None:
        entry = self.store.get(key)
        return entry[0] if entry else None
    
    def list_keys(self) -> list:
        return list(self.store.keys())

wm = WorkingMemory(3)
for i, fact in enumerate(["goal: deploy app","step: build","step: test","step: deploy"]):
    wm.put(f"k{i}", fact)
    print(f"  After k{i}: {wm.list_keys()}")
print(f"\nWorking memory keys: {wm.list_keys()}")

  After k0: ['k0']
  After k1: ['k0', 'k1']
  After k2: ['k0', 'k1', 'k2']
  Evicted: k0
  After k3: ['k1', 'k2', 'k3']

Working memory keys: ['k1', 'k2', 'k3']


## 上下文窗口的有限性

实际 Agent 系统中，LLM 的 context window 就是工作记忆的硬约束：
- GPT-4: 128K tokens
- Claude: 200K tokens
- Gemini: 2M tokens

但即使窗口很大，长上下文也会带来噪声、高延迟和高成本。**Context 管理**是 Agent 工程的核心问题。

> **检查点**：能列举 3 种 context compaction 策略（summarization、chunking、state extraction）。

## 真实 API 实验

LLM 从对话中抽取值得记住的事实（memory extraction），这正是 Mem0 等系统 ADD 管线的第一步。


In [3]:
# 真实记忆抽取：对话 → 结构化事实
dialogue = """用户：我下周二要飞去上海出差三天，住在静安区。
助手：好的，需要我帮你查酒店吗？
用户：不用，公司协议酒店已订好。我对花生过敏，订机票时注意餐食。"""

facts = chat_json(client, f"""从对话中抽取需要长期记住的事实。
返回 JSON: {{"facts": [{{"fact": "...", "type": "preference|plan|constraint", "expires": "temporary|permanent"}}]}}

对话：
{dialogue}""")
for f in facts.get("facts", []):
    print(f"[{f.get('type')}/{f.get('expires')}] {f.get('fact')}")
print("\n观察：LLM 能区分「下周出差」（temporary）与「花生过敏」（permanent constraint）——")
print("这正是记忆系统需要的时效性标注（temporal metadata）。")


[plan/temporary] 用户下周二要去上海出差三天，住在静安区
[constraint/permanent] 用户对花生过敏

观察：LLM 能区分「下周出差」（temporary）与「花生过敏」（permanent constraint）——
这正是记忆系统需要的时效性标注（temporal metadata）。


## 练习

1. 给 WorkingMemory 增加 LRU（Least Recently Used）驱逐策略，替换当前的 FIFO。
2. 实现一个 `compactor(messages, max_tokens)` 函数：当消息历史超过限制时，自动压缩旧消息。
3. 设计一个记忆系统的数据模型：同时支持「当前 task 状态」「用户偏好」「过去失败教训」三种信息的存取。

> **检查点**：能画出你的 Agent 应用的记忆架构图，标注每种记忆的存储介质和生命周期。